In [54]:
import os
import sys
import json
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict

# Наукові обчислення та обробка даних
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Computer Vision та обробка зображень
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pycocotools import mask as maskUtils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# PyTorch - основний фреймворк
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import nms, box_iou
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
# Аугментації
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Перевірка CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
print(f"Training device: {device}")

PyTorch version: 2.9.0+cu128
CUDA available: True
CUDA device: Tesla T4
CUDA version: 12.8
Training device: cuda


In [55]:
@dataclass
class Config:
    coco_root: str = 'coco2017/2/coco2017'
    train_root: str = os.path.join(coco_root, 'train2017')
    val_root: str = os.path.join(coco_root, 'val2017')
    train_ann: str = os.path.join(coco_root, 'annotations', 'instances_train2017.json')
    val_ann: str = os.path.join(coco_root, 'annotations', 'instances_val2017.json')

    # Device
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed: int = 42
    # Checkpoint
    checkpoint_dir: str = './checkpoints'
    save_every: int = 5  # epochs

cfg = Config()
print("\n" + "="*80)
print("КОНФІГУРАЦІЯ ПРОЕКТУ")
print("="*80)
print(f"Dataset: COCO 2017")
print(f"Device: {cfg.device}")
print(f"Checkpoint dir: {cfg.checkpoint_dir}")
print("="*80 + "\n")


КОНФІГУРАЦІЯ ПРОЕКТУ
Dataset: COCO 2017
Device: cuda
Checkpoint dir: ./checkpoints



In [56]:
from typing import Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

"""
Враховуючи аналіз з Частини 1 (lab1.ipynb):
- 80 класів COCO
- Середній розмір bbox: ~118x118 пікселів
- Aspect ratio об'єктів: переважно 0.5-2.0
- Багато малих об'єктів (area < 32²)
"""

class AlexNetBackbone(nn.Module):

    def __init__(self, pretrained: bool = True):
        super().__init__()
        if pretrained:
            alexnet = torchvision.models.alexnet(weights=torchvision.models.AlexNet_Weights.DEFAULT)
        else:
            alexnet = torchvision.models.alexnet(weights=None)
        self.features = alexnet.features
        self.extra_conv = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.out_channels = 256

    def forward(self, x):
        x = self.features(x)
        x = self.extra_conv(x)
        return x


class RegionProposalNetwork(nn.Module):

    def __init__(
        self,
        in_channels: int = 256,
        num_anchors: int = 9,
        feature_stride: int = 32
    ):
        super().__init__()
        self.feature_stride = feature_stride
        self.num_anchors = num_anchors
        self.conv = nn.Conv2d(in_channels, 512, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.cls_logits = nn.Conv2d(512, num_anchors * 2, kernel_size=1)
        self.bbox_pred = nn.Conv2d(512, num_anchors * 4, kernel_size=1)
        for layer in [self.conv, self.cls_logits, self.bbox_pred]:
            nn.init.normal_(layer.weight, std=0.01)
            nn.init.constant_(layer.bias, 0)

    def forward(self, features):
        x = self.conv(features)
        x = self.relu(x)
        objectness = self.cls_logits(x)
        bbox_deltas = self.bbox_pred(x)
        return objectness, bbox_deltas


class DetectionHead(nn.Module):

    def __init__(
        self,
        in_channels: int = 256,
        num_classes: int = 81,
        roi_size: int = 7
    ):
        super().__init__()
        self.num_classes = num_classes
        self.roi_pool = torchvision.ops.RoIPool(
            output_size=(roi_size, roi_size),
            spatial_scale=1.0/32
        )
        fc_input_size = in_channels * roi_size * roi_size
        self.fc6 = nn.Linear(fc_input_size, 4096)
        self.fc7 = nn.Linear(4096, 4096)
        self.cls_score = nn.Linear(4096, num_classes)
        self.bbox_pred = nn.Linear(4096, num_classes * 4)
        self.dropout = nn.Dropout(0.5)

    def forward(self, features, proposals):
        pooled = self.roi_pool(features, proposals)
        pooled = pooled.flatten(start_dim=1)
        x = F.relu(self.fc6(pooled))
        x = self.dropout(x)
        x = F.relu(self.fc7(x))
        x = self.dropout(x)
        cls_scores = self.cls_score(x)
        bbox_deltas = self.bbox_pred(x)
        return cls_scores, bbox_deltas


class AlexNetDetector(nn.Module):

    def __init__(
        self,
        num_classes: int = 81,
        pretrained_backbone: bool = True,
        anchor_scales: Tuple[float, ...] = (32, 64, 128, 256, 512),
        anchor_ratios: Tuple[float, ...] = (0.5, 1.0, 2.0),
    ):
        super().__init__()
        self.num_classes = num_classes
        self.anchor_scales = anchor_scales
        self.anchor_ratios = anchor_ratios
        self.num_anchors = len(anchor_scales) * len(anchor_ratios)
        self.backbone = AlexNetBackbone(pretrained=pretrained_backbone)
        self.rpn = RegionProposalNetwork(
            in_channels=self.backbone.out_channels,
            num_anchors=self.num_anchors,
            feature_stride=32
        )
        self.detection_head = DetectionHead(
            in_channels=self.backbone.out_channels,
            num_classes=num_classes
        )
        print(f"✓ AlexNetDetector initialized:")
        print(f"  - Backbone: AlexNet (pretrained={pretrained_backbone})")
        print(f"  - Num classes: {num_classes}")
        print(f"  - Anchors: {self.num_anchors} = {len(anchor_scales)} scales × {len(anchor_ratios)} ratios")

    def forward(self, images, targets=None):
        if isinstance(images, list):
            images = torch.stack(images)
        features = self.backbone(images)
        objectness, bbox_deltas = self.rpn(features)
        if self.training:
            assert targets is not None, "Targets required for training"
            losses = self._compute_losses(objectness, bbox_deltas, features, targets)
            return losses
        else:
            predictions = self._generate_predictions(objectness, bbox_deltas, features)
            return predictions

    def _compute_losses(self, objectness, bbox_deltas, features, targets):
        losses = {
            'loss_objectness': torch.tensor(0.0, device=objectness.device),
            'loss_rpn_box_reg': torch.tensor(0.0, device=bbox_deltas.device),
            'loss_classifier': torch.tensor(0.0, device=features.device),
            'loss_box_reg': torch.tensor(0.0, device=features.device),
        }
        return losses

    def _generate_predictions(self, objectness, bbox_deltas, features):
        batch_size = features.shape[0]
        predictions = []
        for _ in range(batch_size):
            predictions.append({
                'boxes': torch.empty((0, 4), device=features.device),
                'labels': torch.empty((0,), dtype=torch.int64, device=features.device),
                'scores': torch.empty((0,), device=features.device),
            })
        return predictions


def create_alexnet_detector(num_classes: int = 81, pretrained: bool = True) -> AlexNetDetector:
    model = AlexNetDetector(
        num_classes=num_classes,
        pretrained_backbone=pretrained,
        anchor_scales=(32, 64, 128, 256, 512),
        anchor_ratios=(0.5, 1.0, 2.0),
    )
    return model



print("\n" + "="*80)
print("ТЕСТУВАННЯ ALEXNET DETECTOR")
print("="*80 + "\n")

model_alexnet = create_alexnet_detector(num_classes=81, pretrained=True)
model_alexnet = model_alexnet.to(cfg.device)
model_alexnet.eval()


test_batch = 2
test_images = torch.randn(test_batch, 3, 800, 1200).to(cfg.device)

print(f"Input shape: {test_images.shape}")

with torch.no_grad():
    features = model_alexnet.backbone(test_images)
    print(f"Backbone output: {features.shape}")
    objectness, bbox_deltas = model_alexnet.rpn(features)
    print(f"RPN objectness: {objectness.shape}")
    print(f"RPN bbox_deltas: {bbox_deltas.shape}")

total_params = sum(p.numel() for p in model_alexnet.parameters())
trainable_params = sum(p.numel() for p in model_alexnet.parameters() if p.requires_grad)

print(f"\nСтатистика моделі:")
print(f"  - Всього параметрів: {total_params:,}")
print(f"  - Trainable параметрів: {trainable_params:,}")
print(f"  - Розмір моделі: ~{total_params * 4 / 1024 / 1024:.2f} MB (float32)")

print("\n" + "="*80)
print("✓ AlexNet Detector успішно створено!")
print("="*80 + "\n")



ТЕСТУВАННЯ ALEXNET DETECTOR

✓ AlexNetDetector initialized:
  - Backbone: AlexNet (pretrained=True)
  - Num classes: 81
  - Anchors: 15 = 5 scales × 3 ratios
Input shape: torch.Size([2, 3, 800, 1200])
Backbone output: torch.Size([2, 256, 24, 36])
RPN objectness: torch.Size([2, 30, 24, 36])
RPN bbox_deltas: torch.Size([2, 60, 24, 36])

Статистика моделі:
  - Всього параметрів: 74,701,103
  - Trainable параметрів: 74,701,103
  - Розмір моделі: ~284.96 MB (float32)

✓ AlexNet Detector успішно створено!



In [57]:
import torch
import torch.nn as nn
import torchvision
from torchvision.ops import roi_align as _roi_align
from typing import Tuple
def _roi_align_call(feats, boxes, *, output_size, spatial_scale, sampling_ratio=-1, aligned=True):

    try:
        return _roi_align(
            feats, boxes,
            output_size=output_size,
            spatial_scale=spatial_scale,
            sampling_ratio=sampling_ratio,
            aligned=aligned,
        )
    except TypeError:

        return _roi_align(
            feats, boxes,
            output_size=output_size,
            spatial_scale=spatial_scale,
            sampling_ratio=sampling_ratio,
        )

def roi_align_compat(feats, rois, output_size, spatial_scale, sampling_ratio=-1, aligned=True):
    if isinstance(rois, (list, tuple)):
        return _roi_align_call(
            feats, rois,
            output_size=output_size,
            spatial_scale=spatial_scale,
            sampling_ratio=sampling_ratio,
            aligned=aligned,
        )

    if isinstance(rois, torch.Tensor):

        try:
            return _roi_align_call(
                feats, rois,
                output_size=output_size,
                spatial_scale=spatial_scale,
                sampling_ratio=sampling_ratio,
                aligned=aligned,
            )
        except (TypeError, RuntimeError):

            B = feats.shape[0]
            boxes = [rois[rois[:, 0] == b, 1:] for b in range(B)]
            return _roi_align_call(
                feats, boxes,
                output_size=output_size,
                spatial_scale=spatial_scale,
                sampling_ratio=sampling_ratio,
                aligned=aligned,
            )

    raise TypeError("rois must be Tensor[N,5] or List[Tensor[K,4]]")

class SimpleRCNN(nn.Module):
    def __init__(self, num_classes: int = 81, pretrained_backbone: bool = True, roi_size: int = 7):
        super().__init__()
        alex = torchvision.models.alexnet(
            weights=torchvision.models.AlexNet_Weights.DEFAULT if pretrained_backbone else None
        )
        self.features = alex.features
        self.feature_dim = 256 * roi_size * roi_size


        self.roi_output_size = (roi_size, roi_size)
        self.spatial_scale = 1.0 / 32.0


        self.fc_layers = nn.Sequential(
            nn.Linear(self.feature_dim, 4096),
            nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(0.5),
            nn.Linear(4096, num_classes)
        )

    def forward(self, images: torch.Tensor, rois: torch.Tensor) -> torch.Tensor:
        feats = self.features(images)
        pooled = roi_align_compat(
    feats, rois,
    output_size=self.roi_output_size,
    spatial_scale=self.spatial_scale,
    aligned=True
)
        pooled = pooled.flatten(start_dim=1)
        logits = self.fc_layers(pooled)
        return logits



test_model = SimpleRCNN(num_classes=81).to(cfg.device)
test_images = torch.randn(1, 3, 800, 800, device=cfg.device)
test_rois = torch.tensor([[0, 50, 60, 200, 300]], dtype=torch.float32, device=cfg.device)

with torch.no_grad():
    out = test_model(test_images, test_rois)
print("R-CNN output shape:", out.shape)


R-CNN output shape: torch.Size([1, 81])


In [58]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class TinyYOLO(nn.Module):
    """
    Спрощена YOLO v1 Tiny:
    - підтримує довільний розмір входу (напр. 224 або 448)
    - вихід: [B, S, S, B*5 + C]
    """
    def __init__(self, num_classes: int = 20, S: int = 7, B: int = 2):
        super().__init__()
        self.S, self.B, self.C = S, B, num_classes

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.LeakyReLU(0.1), nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, 3, padding=1), nn.LeakyReLU(0.1), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.LeakyReLU(0.1), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.LeakyReLU(0.1), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.LeakyReLU(0.1), nn.MaxPool2d(2, 2),
        )


        self.to_grid = nn.AdaptiveAvgPool2d((S, S))


        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * (S * S), 4096),
            nn.LeakyReLU(0.1),
            nn.Linear(4096, S * S * (B * 5 + num_classes))
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.features(x)
        grid  = self.to_grid(feats)
        out   = self.head(grid)
        out   = out.view(-1, self.S, self.S, self.B * 5 + self.C)
        return out



yolo = TinyYOLO(num_classes=20, S=7, B=2)

x1 = torch.randn(2, 3, 224, 224)
o1 = yolo(x1)
print("224 ->", o1.shape)

x2 = torch.randn(2, 3, 448, 448)
o2 = yolo(x2)
print("448 ->", o2.shape)


224 -> torch.Size([2, 7, 7, 30])
448 -> torch.Size([2, 7, 7, 30])


In [59]:
import torch
import torch.nn as nn

EPS = 1e-6


def box_iou(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    """
    Обчислює IoU між двома наборами боксів.
    boxes1: [N,4] у форматі (x1,y1,x2,y2)
    boxes2: [M,4] у форматі (x1,y1,x2,y2)
    Повертає: [N,M] IoU
    """
    if boxes1.numel() == 0 or boxes2.numel() == 0:

        return torch.zeros((boxes1.shape[0], boxes2.shape[0]), dtype=boxes1.dtype, device=boxes1.device)


    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)


    inter_x1 = torch.max(boxes1[:, 0].unsqueeze(1), boxes2[:, 0].unsqueeze(0))
    inter_y1 = torch.max(boxes1[:, 1].unsqueeze(1), boxes2[:, 1].unsqueeze(0))
    inter_x2 = torch.min(boxes1[:, 2].unsqueeze(1), boxes2[:, 2].unsqueeze(0))
    inter_y2 = torch.min(boxes1[:, 3].unsqueeze(1), boxes2[:, 3].unsqueeze(0))

    inter_w = (inter_x2 - inter_x1).clamp(min=0)
    inter_h = (inter_y2 - inter_y1).clamp(min=0)
    inter_area = inter_w * inter_h

    union_area = area1.unsqueeze(1) + area2.unsqueeze(0) - inter_area
    return inter_area / (union_area + EPS)


def mean_average_precision(
    pred_boxes,
    true_boxes,
    iou_threshold: float = 0.5,
):
    """
    Обчислення mAP по класах.

    Формати:
      pred_boxes: List[ (img_id:int, cls:int, score:float, x1:float, y1:float, x2:float, y2:float) ]
      true_boxes: List[ (img_id:int, cls:int,            x1:float, y1:float, x2:float, y2:float) ]

    Повертає:
      mAP (torch.tensor float)
    """

    classes = sorted({gt[1] for gt in true_boxes})
    if len(classes) == 0:
        return torch.tensor(0.0)

    average_precisions = []

    from collections import defaultdict
    gt_index = {c: defaultdict(list) for c in classes}
    for (img_id, cls, x1, y1, x2, y2) in true_boxes:
        gt_index[cls][img_id].append([x1, y1, x2, y2])
    for c in classes:
        for img_id in gt_index[c].keys():
            boxes = torch.tensor(gt_index[c][img_id], dtype=torch.float32)
            gt_index[c][img_id] = {
                "boxes": boxes,
                "matched": torch.zeros((boxes.shape[0],), dtype=torch.bool)
            }


    for c in classes:

        detections_c = [d for d in pred_boxes if d[1] == c]
        detections_c.sort(key=lambda x: x[2], reverse=True)

        n_gt = sum(gt_index[c][img_id]["boxes"].shape[0] for img_id in gt_index[c].keys())
        if n_gt == 0:

            continue

        TP = torch.zeros((len(detections_c),), dtype=torch.float32)
        FP = torch.zeros((len(detections_c),), dtype=torch.float32)

        for i, (img_id, _, score, x1, y1, x2, y2) in enumerate(detections_c):

            if img_id not in gt_index[c]:
                FP[i] = 1.0
                continue

            gt_boxes = gt_index[c][img_id]["boxes"]
            matched = gt_index[c][img_id]["matched"]

            if gt_boxes.numel() == 0:
                FP[i] = 1.0
                continue


            pb = torch.tensor([[x1, y1, x2, y2]], dtype=torch.float32)
            ious = box_iou(pb, gt_boxes).squeeze(0)

            best_iou, best_idx = (ious.max().item(), int(ious.argmax().item())) if ious.numel() > 0 else (0.0, -1)

            if best_iou >= iou_threshold and not matched[best_idx]:
                TP[i] = 1.0
                matched[best_idx] = True
            else:
                FP[i] = 1.0

        tp_cum = torch.cumsum(TP, dim=0)
        fp_cum = torch.cumsum(FP, dim=0)
        recalls = tp_cum / (n_gt + EPS)
        precisions = tp_cum / (tp_cum + fp_cum + EPS)

        if precisions.numel() > 0:
            mpre = precisions.clone()
            for k in range(mpre.numel() - 2, -1, -1):
                mpre[k] = torch.maximum(mpre[k], mpre[k + 1])
            ap = torch.trapz(mpre, recalls).item()
        else:
            ap = 0.0

        average_precisions.append(ap)

    if len(average_precisions) == 0:
        return torch.tensor(0.0)

    return torch.tensor(sum(average_precisions) / len(average_precisions), dtype=torch.float32)


In [60]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DetectionLoss(nn.Module):
    """
    Узагальнений loss для детекції:
      - Класифікація: CrossEntropy(ignore_index)
      - Локація: SmoothL1 лише для позитивів (true_cls > 0)
      - Підтримує pred_locs формату [N,4] або [N, C*4] (пер-класова регресія)
    """
    def __init__(
        self,
        alpha: float = 1.0,
        beta: float = 1.0,
        ignore_index: int = -1,
        smoothl1_beta: float = 1.0
    ):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.ignore_index = ignore_index
        self.loc_loss_fn = nn.SmoothL1Loss(reduction="sum", beta=smoothl1_beta)
        self.cls_loss_fn = nn.CrossEntropyLoss(ignore_index=ignore_index)

    def forward(
        self,
        pred_locs: torch.Tensor,
        true_locs: torch.Tensor,
        pred_cls: torch.Tensor,
        true_cls: torch.Tensor
    ):
        device = pred_cls.device
        N = pred_cls.size(0)
        C = pred_cls.size(1)


        cls_loss = self.cls_loss_fn(pred_cls, true_cls)

        pos_mask = (true_cls > 0) & (true_cls != self.ignore_index)
        num_pos = int(pos_mask.sum().item())

        if num_pos > 0:

            if pred_locs.size(1) == 4:
                loc_pred_pos = pred_locs[pos_mask]
            elif pred_locs.size(1) == C * 4:

                loc_all = pred_locs.view(N, C, 4)
                cls_pos = true_cls[pos_mask]

                loc_pred_pos = loc_all[pos_mask, cls_pos]
            else:
                raise ValueError(f"pred_locs has invalid shape {tuple(pred_locs.shape)}; expected [N,4] or [N,{C*4}]")

            loc_true_pos = true_locs[pos_mask]

            loc_loss = self.loc_loss_fn(loc_pred_pos, loc_true_pos) / max(1, num_pos)
        else:

            loc_loss = torch.tensor(0.0, device=device)

        total = self.alpha * loc_loss + self.beta * cls_loss

        return total, {
            "loc_loss": float(loc_loss.item()),
            "cls_loss": float(cls_loss.item()),
            "num_pos":  num_pos
        }


In [61]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SegmentationHead1x1(nn.Module):
    def __init__(self, in_channels: int, num_classes: int, upsample_scale: int = 32):
        super().__init__()
        self.score = nn.Conv2d(in_channels, num_classes, kernel_size=1)
        self.upsample_scale = upsample_scale

    def forward(self, fmap: torch.Tensor, out_size: tuple[int, int] | None = None) -> torch.Tensor:
        x = self.score(fmap)
        if out_size is not None:
            return F.interpolate(x, size=out_size, mode='bilinear', align_corners=False)
        H, W = fmap.shape[-2] * self.upsample_scale, fmap.shape[-1] * self.upsample_scale
        return F.interpolate(x, size=(H, W), mode='bilinear', align_corners=False)


class AlexNetSegmentationModel(nn.Module):
    def __init__(self, num_classes: int, pretrained_backbone: bool = True):
        super().__init__()
        self.backbone = AlexNetBackbone(pretrained=pretrained_backbone)
        assert hasattr(self.backbone, "out_channels"), "AlexNetBackbone must have .out_channels"
        self.head = SegmentationHead1x1(
            in_channels=self.backbone.out_channels,
            num_classes=num_classes,
            upsample_scale=32
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        H, W = x.shape[-2:]
        fmap = self.backbone(x)
        logits = self.head(fmap, out_size=(H, W))
        return logits


@torch.no_grad()
def extract_pixel_features(model_backbone: nn.Module,
                           images: torch.Tensor,
                           device: str | torch.device = "cuda"):
    model_backbone.eval()
    images = images.to(device, non_blocking=True)
    fmap = model_backbone(images)
    fmap_up = F.interpolate(fmap, size=images.shape[-2:], mode='bilinear', align_corners=False)
    B, C, H, W = fmap_up.shape
    feat = fmap_up.permute(0, 2, 3, 1).contiguous().view(B * H * W, C)
    return feat.detach().cpu().numpy()


In [62]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

class FCN32s(nn.Module):
    def __init__(self, num_classes: int = 21, pretrained_backbone: bool = True, freeze_encoder: bool = False):
        super().__init__()
        alex = torchvision.models.alexnet(
            weights=torchvision.models.AlexNet_Weights.IMAGENET1K_V1 if pretrained_backbone else None
        )

        self.encoder = alex.features

        self.enc_out_channels = 256 if not hasattr(self.encoder[-1], "out_channels") else self.encoder[-1].out_channels
        self.score = nn.Conv2d(self.enc_out_channels, num_classes, kernel_size=1)

        if freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        H, W = x.shape[-2:]
        feat = self.encoder(x)
        score = self.score(feat)
        out = F.interpolate(score, size=(H, W), mode='bilinear', align_corners=False)
        return out


In [63]:
import torch
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models import ResNet50_Weights

def get_maskrcnn(
    num_classes: int,
    pretrained_backbone: bool = True,
    pretrained_weights: bool = True,
    box_score_thresh: float | None = None,
    batch_size_per_image: int | None = None,
):
    """
    Повертає Mask R-CNN (ResNet50-FPN) з оновленим API weights/weights_backbone
    та підміненою box/mask головами під num_classes.
    """
    if pretrained_weights:
        model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    else:
        model = maskrcnn_resnet50_fpn(
            weights=None,
            weights_backbone=(ResNet50_Weights.IMAGENET1K_V1 if pretrained_backbone else None),
        )


    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
        in_features, num_classes
    )


    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = torchvision.models.detection.mask_rcnn.MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )


    if box_score_thresh is not None and hasattr(model.roi_heads, "score_thresh"):
        model.roi_heads.score_thresh = float(box_score_thresh)
    if batch_size_per_image is not None:
        model.roi_heads.batch_size_per_image = int(batch_size_per_image)

    return model


In [64]:

DEFAULTS_MINI = {
    "train": {
        "optimizer": "Adam",
        "learning_rate": 1e-3,
        "weight_decay": 1e-4,
        "batch_size": 8,
        "epochs": 10,
        "scheduler": "none",
        "grad_clip": None,
        "seed": 42,

        "momentum": 0.9,
        "nesterov": True,
    },
    "aug": {
        "hflip_p": 0.5,
    }
}


SEARCH_SPACE_MINI = {
    "train": {
        "learning_rate": [1e-3, 3e-4],
        "batch_size":    [4, 8],
        "weight_decay":  [0.0, 1e-4],
        "scheduler":     ["none", "StepLR_5_0.5"],
    },

    "alexnet_detector": {
        "anchor_scales": [(32, 64, 128)],
        "anchor_ratios": [(0.5, 1.0, 2.0)],
        "rpn_nms_thresh": [0.7],
        "roi_size":       [7],
    },
    "rcnn": {
        "roi_size":        [7],
        "freeze_backbone": [True, False],
    },
    "tiny_yolo": {
        "S":           [7],
        "B":           [1, 2],
        "image_size":  [224],
        "conf_thresh": [0.25, 0.4],
    },

    "fcn32s": {
        "upsample_scale": [32],
        "loss_type":      ["CE"],    #
    },
    "mask_rcnn": {
        "box_score_thresh":     [0.5],
        "batch_size_per_image": [256],
    }
}


import copy, re, torch
from torch.optim import Adam, AdamW, SGD
from torch.optim.lr_scheduler import StepLR

def set_seed(seed: int = 42):
    import os, random, numpy as np
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def hp_compose(model_key: str, overrides: dict | None = None):
    """
    Беремо DEFAULTS_MINI + перші значення з SEARCH_SPACE_MINI[model_key].
    overrides (необов’язково): {"train": {...}, model_key: {...}}
    """
    hp = copy.deepcopy(DEFAULTS_MINI)
    if model_key in SEARCH_SPACE_MINI:
        hp[model_key] = {
            k: (v[0] if isinstance(v, list) else v)
            for k, v in SEARCH_SPACE_MINI[model_key].items()
        }
    if overrides:
        if "train" in overrides:
            hp["train"].update(overrides["train"])
        if model_key in overrides:
            hp[model_key].update(overrides[model_key])
    set_seed(hp["train"]["seed"])
    return hp

def build_optimizer(model: torch.nn.Module, hp: dict):
    t = hp["train"]
    name = str(t.get("optimizer", "Adam")).lower()
    if name == "adam":
        return Adam(model.parameters(), lr=t["learning_rate"], weight_decay=t["weight_decay"])
    if name == "adamw":
        return AdamW(model.parameters(), lr=t["learning_rate"], weight_decay=t["weight_decay"])
    if name == "sgd":
        return SGD(
            model.parameters(),
            lr=t["learning_rate"],
            momentum=float(t.get("momentum", 0.9)),
            nesterov=bool(t.get("nesterov", True)),
            weight_decay=t["weight_decay"],
        )
    raise ValueError(f"Unknown optimizer: {t['optimizer']}")

def build_scheduler(optimizer, hp: dict):
    sch = str(hp["train"].get("scheduler", "none")).lower()
    if sch.startswith("steplr_"):

        m = re.match(r"steplr_(\d+)_([0-9]*\.?[0-9]+)", sch)
        step_size = int(m.group(1)) if m else 5
        gamma = float(m.group(2)) if m else 0.5
        return StepLR(optimizer, step_size=step_size, gamma=gamma)
    return None


In [65]:

import time, itertools, pandas as pd, torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

def _make_grid(space: dict):
    """Перетворює словник space у декартів добуток значень."""
    if not space:
        return [dict()]
    keys = list(space.keys())
    vals = [v if isinstance(v, list) else [v] for v in (space[k] for k in keys)]
    return [dict(zip(keys, combo)) for combo in itertools.product(*vals)]

def _tensor_mean_abs(x):
    """Акуратно згортає будь-яку структуру до тензорної скалярної «втрати»."""
    if torch.is_tensor(x):
        return x.float().abs().mean()
    if isinstance(x, dict):
        parts = [v for v in x.values() if torch.is_tensor(v)]
        if parts:
            return torch.stack([p.float().abs().mean() for p in parts]).mean()
        return torch.tensor(0.0, device=cfg.device)
    if isinstance(x, (list, tuple)):
        parts = [p for p in x if torch.is_tensor(p)]
        if parts:
            return torch.stack([p.float().abs().mean() for p in parts]).mean()

        vals = [_tensor_mean_abs(p) for p in x]
        return torch.stack(vals).mean() if vals else torch.tensor(0.0, device=cfg.device)
    return torch.tensor(0.0, device=cfg.device)

def _proxy_step(model, optimizer, scheduler, hp, model_key: str):
    """Один міні-крок 'навчання' + проксі-метрика (–loss). Замінюй на реальний train/eval."""
    bs = hp["train"]["batch_size"]
    img_size = 224 if model_key != "tiny_yolo" else hp.get(model_key, {}).get("image_size", 224)
    xb = torch.randn(bs, 3, img_size, img_size, device=cfg.device)

    model.train()
    optimizer.zero_grad(set_to_none=True)


    out = None
    try:
        out = model(xb)
    except Exception:
        out = None

    loss = None
    if out is not None:
        loss = _tensor_mean_abs(out)

        if not getattr(loss, "requires_grad", False):
            out = model.backbone(xb) if hasattr(model, "backbone") else xb
            loss = _tensor_mean_abs(out)
    else:

        out = model.backbone(xb) if hasattr(model, "backbone") else xb
        loss = _tensor_mean_abs(out)

    loss.backward()
    if hp["train"].get("grad_clip"):
        torch.nn.utils.clip_grad_norm_(model.parameters(), hp["train"]["grad_clip"])
    optimizer.step()
    if scheduler:
        scheduler.step()

    metric = float(-loss.item())
    return metric

def tune_hparams(model_key: str, model_factory,
                 train_space: dict | None = None,
                 model_space: dict | None = None,
                 max_trials: int | None = None,
                 repeats: int = 1):
    """
    model_key: "alexnet_detector" | "rcnn" | "tiny_yolo" | "fcn32s" | "mask_rcnn"
    model_factory: lambda що повертає НОВУ модель (на кожен конфіг)
    train_space: підпростір із SEARCH_SPACE_MINI["train"] (за замовчуванням: LR, batch_size, weight_decay, scheduler)
    model_space: підпростір із SEARCH_SPACE_MINI[model_key]
    max_trials: обмежити загальну к-сть проб
    repeats: скільки раз повторювати кожен конфіг для стабільності
    """

    if train_space is None:
        base = SEARCH_SPACE_MINI["train"]
        train_space = {k: base[k] for k in ["learning_rate", "batch_size", "weight_decay", "scheduler"] if k in base}
    if model_space is None:
        model_space = SEARCH_SPACE_MINI.get(model_key, {})


    grid_train  = _make_grid(train_space)
    grid_model  = _make_grid(model_space)
    combos = list(itertools.product(grid_train, grid_model))
    if max_trials is not None:
        combos = combos[:max_trials]


    records = []
    for t_cfg, m_cfg in combos:
        overrides = {"train": t_cfg, model_key: m_cfg}
        hp = hp_compose(model_key, overrides=overrides)
        model = model_factory().to(cfg.device)
        opt   = build_optimizer(model, hp)
        sch   = build_scheduler(opt, hp)

        metrics = []
        for _ in range(repeats):
            m = _proxy_step(model, opt, sch, hp, model_key)
            metrics.append(m)
        metric_mean = float(np.mean(metrics))
        rec = {"metric": metric_mean}
        rec.update({f"train.{k}": v for k, v in t_cfg.items()})
        rec.update({f"{model_key}.{k}": v for k, v in m_cfg.items()})
        records.append(rec)


        del model, opt, sch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    df = pd.DataFrame.from_records(records).sort_values("metric", ascending=False).reset_index(drop=True)


    importance = []
    param_cols = [c for c in df.columns if c != "metric"]
    for col in param_cols:
        grp = df.groupby(col)["metric"].mean()
        if len(grp) >= 2:
            delta = float(grp.max() - grp.min())
            importance.append({
                "param": col,
                "importance_delta": delta,
                "best_value": grp.idxmax(),
                "worst_value": grp.idxmin()
            })
    imp_df = pd.DataFrame(importance).sort_values("importance_delta", ascending=False).reset_index(drop=True)

    print("=== Топ конфігів ===")
    display(df.head(10))
    print("\n=== Важливість гіперпараметрів (Δ) ===")
    display(imp_df)


    if not imp_df.empty:
        ax = imp_df.set_index("param")["importance_delta"].plot(kind="barh", figsize=(6,4))
        ax.invert_yaxis()
        ax.set_xlabel("Δ metric (більше = важливіше)")
        ax.set_ylabel("гіперпараметр")
        plt.tight_layout()
        plt.show()

    return df, imp_df


In [66]:

from pycocotools.coco import COCO
from pycocotools import mask as maskUtils
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2, numpy as np, os, torch

class COCOMaskDataset(Dataset):
    def __init__(self, img_root: str, ann_file: str, hflip_p: float = 0.5):
        """
        Повертає:
          image: FloatTensor [3,H,W] (нормалізований)
          target: dict(
              boxes:  FloatTensor [N,4] (x1,y1,x2,y2)
              labels: LongTensor  [N]
              masks:  UInt8Tensor [N,H,W] (0/1)
              image_id: LongTensor [1]
          )
        """
        super().__init__()
        self.coco = COCO(ann_file)
        self.img_root = img_root
        self.img_ids = self.coco.getImgIds()


        self.cat_ids = sorted(self.coco.getCatIds())
        self.cat_id_to_idx = {cid: i+1 for i, cid in enumerate(self.cat_ids)}
        self.num_classes = len(self.cat_ids) + 1


        self.tf = A.Compose([
            A.HorizontalFlip(p=hflip_p),
            A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
            ToTensorV2(),
        ], is_check_shapes=False)

    def __len__(self):
        return len(self.img_ids)

    def _load_image(self, img_info):
        path = os.path.join(self.img_root, img_info['file_name'])
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img

    def _ann_to_mask(self, ann, h, w):
        seg = ann.get('segmentation', None)
        if isinstance(seg, list):  # полігони
            rles = maskUtils.frPyObjects(seg, h, w)
            m = maskUtils.decode(rles)
            if m.ndim == 3:
                m = np.any(m, axis=2)
        elif isinstance(seg, dict):  # RLE
            m = maskUtils.decode(seg)
            if m.ndim == 3:
                m = m[..., 0]
        else:

            m = np.zeros((h, w), dtype=np.uint8)
            x, y, w2, h2 = ann['bbox']
            x1, y1, x2, y2 = int(x), int(y), int(x + w2), int(y + h2)
            m[max(y1,0):max(y2,0), max(x1,0):max(x2,0)] = 1
        return m.astype(np.uint8)

    def __getitem__(self, idx: int):
        img_id = self.img_ids[idx]
        info = self.coco.loadImgs(img_id)[0]
        h, w = info['height'], info['width']
        img = self._load_image(info)

        ann_ids = self.coco.getAnnIds(imgIds=[img_id], iscrowd=None)
        anns = self.coco.loadAnns(ann_ids)


        masks_np, labels = [], []
        for ann in anns:
            if ann.get('iscrowd', 0) == 1:
                continue
            labels.append(self.cat_id_to_idx[ann['category_id']])
            masks_np.append(self._ann_to_mask(ann, h, w))

        if len(masks_np) == 0:
            masks_stack = np.zeros((0, h, w), dtype=np.uint8)
        else:

            masks_stack = np.ascontiguousarray(np.stack(masks_np, axis=0), dtype=np.uint8)



        tf_res = self.tf(image=img, masks=list(masks_stack))
        img_t = tf_res['image']
        masks_aug = tf_res['masks']


        pairs = []
        for m, lab in zip(masks_aug, labels):
            m_np = np.asarray(m, dtype=np.uint8)
            if m_np.any():
                pairs.append((m_np, lab))

        boxes = []
        if pairs:
            masks_re, labels_re = zip(*pairs)
            labels = list(labels_re)
            masks_t = np.ascontiguousarray(np.stack(masks_re, axis=0), dtype=np.uint8)  # [N,H,W]
            for m_np in masks_re:
                ys, xs = np.where(m_np > 0)
                x1, y1, x2, y2 = xs.min(), ys.min(), xs.max(), ys.max()
                if x2 == x1: x2 = x1 + 1
                if y2 == y1: y2 = y1 + 1
                boxes.append([float(x1), float(y1), float(x2), float(y2)])
        else:
            H, W = img_t.shape[1], img_t.shape[2]
            masks_t = np.zeros((0, H, W), dtype=np.uint8)
            labels = []
            boxes = []

        boxes_t = torch.as_tensor(boxes, dtype=torch.float32)
        if boxes_t.numel() == 0:
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)

        target = {
            "boxes":  boxes_t,
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "masks":  torch.as_tensor(masks_t, dtype=torch.uint8),
            "image_id": torch.tensor([img_id], dtype=torch.int64),
        }
        return img_t, target


def collate_fn(batch):
    imgs, targets = list(zip(*batch))
    return list(imgs), list(targets)


In [67]:

from IPython.display import display
from typing import List, Dict, Any, Optional
import pandas as pd


try:
    History
except NameError:
    class History:
        """Проста історія метрик/втрат із зручним API."""
        __slots__ = ("_rows",)

        def __init__(self) -> None:
            self._rows: List[Dict[str, Any]] = []

        def log(self, **kwargs: Any) -> None:

            self._rows.append(kwargs)

        def to_df(self) -> pd.DataFrame:

            return pd.DataFrame(self._rows, copy=True)

        def last(self) -> Optional[Dict[str, Any]]:
            return self._rows[-1] if self._rows else None

        def reset(self) -> None:
            self._rows.clear()

        def __len__(self) -> int:
            return len(self._rows)


In [68]:

import torch
from torch.nn.utils import clip_grad_norm_

def train_one_epoch_maskrcnn(model, loader, optimizer, device, grad_clip=None):
    model.train()
    running = 0.0
    for images, targets in loader:
        images  = [img.to(device, non_blocking=True) for img in images]
        targets = [{k: (v.to(device) if torch.is_tensor(v) else v)
                    for k, v in t.items()} for t in targets]

        optimizer.zero_grad(set_to_none=True)
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        loss.backward()
        if grad_clip is not None:
            clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        running += float(loss.detach().item())
    return running / max(1, len(loader))

@torch.no_grad()
def eval_one_epoch_maskrcnn(model, loader, device):

    was_training = model.training
    model.train()
    running = 0.0
    for images, targets in loader:
        images  = [img.to(device, non_blocking=True) for img in images]
        targets = [{k: (v.to(device) if torch.is_tensor(v) else v)
                    for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        running += float(loss.detach().item())
    model.train(was_training)
    return running / max(1, len(loader))


In [ ]:

import os
import torch
from torch.utils.data import DataLoader

hp_mask = hp_compose(
    "mask_rcnn",
    overrides={
        "train": {
            "batch_size": 2,      # якщо все одно OOM — став 1
            "learning_rate": 5e-4
        }
    }
)
print("batch_size for mask_rcnn:", hp_mask["train"]["batch_size"])


train_ds = COCOMaskDataset(cfg.train_root, cfg.train_ann, hflip_p=DEFAULTS_MINI["aug"]["hflip_p"])
val_ds   = COCOMaskDataset(cfg.val_root,   cfg.val_ann,   hflip_p=0.0)


g = torch.Generator()
g.manual_seed(hp_mask["train"]["seed"])
common_dl_args = dict(
    batch_size=hp_mask["train"]["batch_size"],
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
    persistent_workers=True
)
train_loader = DataLoader(train_ds, shuffle=True,  generator=g, **common_dl_args)
val_loader   = DataLoader(val_ds,   shuffle=False,               **common_dl_args)


num_classes = train_ds.num_classes
model = get_maskrcnn(num_classes=num_classes,
                     pretrained_backbone=True,
                     pretrained_weights=True).to(cfg.device)


mask_hp = hp_mask.get("mask_rcnn", {})
if "batch_size_per_image" in mask_hp:
    model.roi_heads.batch_size_per_image = int(mask_hp["batch_size_per_image"])
if "box_score_thresh" in mask_hp:

    if hasattr(model.roi_heads, "score_thresh"):
        model.roi_heads.score_thresh = float(mask_hp["box_score_thresh"])
    if hasattr(model.roi_heads, "box_score_thresh"):
        model.roi_heads.box_score_thresh = float(mask_hp["box_score_thresh"])


optimizer = build_optimizer(model, hp_mask)
scheduler = build_scheduler(optimizer, hp_mask)


os.makedirs(cfg.checkpoint_dir, exist_ok=True)
hist = History()
best_val = float("inf")

for epoch in range(1, hp_mask["train"]["epochs"] + 1):
    tr_loss = train_one_epoch_maskrcnn(
        model, train_loader, optimizer, cfg.device,
        grad_clip=hp_mask["train"]["grad_clip"]
    )
    val_loss = eval_one_epoch_maskrcnn(model, val_loader, cfg.device)
    if scheduler:
        scheduler.step()

    hist.log(epoch=epoch, train_loss=tr_loss, val_loss=val_loss, metric=-val_loss)
    print(f"[Epoch {epoch:02d}] train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}")


    is_best = val_loss < best_val
    if is_best:
        best_val = val_loss
    if is_best or (epoch % cfg.save_every == 0):
        ckpt_path = os.path.join(cfg.checkpoint_dir, f"maskrcnn_e{epoch:02d}_val{val_loss:.4f}.pth")
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict() if scheduler else None,
            "hp": hp_mask,
            "best_val": best_val,
        }, ckpt_path)
        print(f"✓ Saved checkpoint to: {ckpt_path}")

df_hist_mask = hist.to_df()
summarize_run(df_hist_mask, hp_mask, save_json_path="best_hparams_maskrcnn.json")


batch_size for mask_rcnn: 2
loading annotations into memory...


In [ ]:

class COCOSemanticDataset(Dataset):
    """
    Формує семантичну маску (H×W) з instance-анотацій COCO:
      фон = 0, класи = 1..K (mapping за cat_id_to_idx).
    При перекриттях пікселі більших об'єктів мають пріоритет (сортування за площею).
    """
    def __init__(self, img_root: str, ann_file: str, hflip_p: float = 0.5):
        super().__init__()
        self.coco = COCO(ann_file)
        self.img_root = img_root
        self.img_ids = self.coco.getImgIds()


        self.cat_ids = sorted(self.coco.getCatIds())
        self.cat_id_to_idx = {cid: i + 1 for i, cid in enumerate(self.cat_ids)}
        self.num_classes = len(self.cat_ids) + 1


        self.tf = A.Compose([
            A.HorizontalFlip(p=hflip_p),
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ], is_check_shapes=False)

    def __len__(self):
        return len(self.img_ids)

    def _load_image(self, info):
        path = os.path.join(self.img_root, info["file_name"])
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(path)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    @staticmethod
    def _safe_bbox_mask(h, w, bbox):
        """Fallback: bbox -> прямокутна маска з клампом у межі зображення."""
        x, y, bw, bh = bbox
        x1, y1 = int(np.floor(x)), int(np.floor(y))
        x2, y2 = int(np.ceil(x + bw)), int(np.ceil(y + bh))
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        m = np.zeros((h, w), dtype=np.uint8)
        if x2 > x1 and y2 > y1:
            m[y1:y2, x1:x2] = 1
        return m

    @staticmethod
    def _to_bool_mask(m):
        """Нормалізує маску з pycocotools до 2-D bool."""
        if m is None:
            return None
        m = np.asarray(m)
        if m.ndim == 3:
            m = m.max(axis=2)
        return np.ascontiguousarray(m.astype(np.uint8)) > 0

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        info = self.coco.loadImgs(img_id)[0]
        h, w = int(info["height"]), int(info["width"])
        img = self._load_image(info)

        ann_ids = self.coco.getAnnIds(imgIds=[img_id], iscrowd=None)
        anns = self.coco.loadAnns(ann_ids)

        anns = sorted(anns, key=lambda a: float(a.get("area", 0.0)))

        sem = np.zeros((h, w), dtype=np.uint16)

        for ann in anns:
            if ann.get("iscrowd", 0) == 1:
                continue
            cid = ann["category_id"]
            cls_idx = self.cat_id_to_idx.get(cid)
            if cls_idx is None:
                continue

            seg = ann.get("segmentation", None)
            m_bool = None
            try:
                if isinstance(seg, list):
                    rles = maskUtils.frPyObjects(seg, h, w)
                    m_bool = self._to_bool_mask(maskUtils.decode(rles))
                elif isinstance(seg, dict):
                    m_bool = self._to_bool_mask(maskUtils.decode(seg))
                else:
                    m_bool = self._to_bool_mask(self._safe_bbox_mask(h, w, ann["bbox"]))
            except Exception:

                m_bool = self._to_bool_mask(self._safe_bbox_mask(h, w, ann["bbox"]))

            if m_bool is not None and m_bool.any():
                sem[m_bool] = cls_idx
        tf_res = self.tf(image=img, mask=sem.astype(np.int64))
        img_t = tf_res["image"]
        mask_t = tf_res["mask"].long().contiguous()
        return img_t, mask_t


In [ ]:

import torch
from typing import Optional, Tuple

def _fast_hist(pred: torch.Tensor,
               target: torch.Tensor,
               num_classes: int) -> torch.Tensor:
    """
    Побудова матриці неточностей (confusion matrix) K×K через bincount.
    pred, target: [H,W] або [N,H,W] (подавати батч поелементно).
    """

    pred   = pred.to(torch.int64)
    target = target.to(torch.int64)

    k = (target >= 0) & (target < num_classes)
    if k.sum() == 0:

        return torch.zeros((num_classes, num_classes), device=target.device, dtype=torch.float32)


    inds = (num_classes * target[k] + pred[k]).to(torch.int64)
    hist = torch.bincount(inds, minlength=num_classes**2)
    return hist.reshape(num_classes, num_classes).to(torch.float32)


@torch.inference_mode()
def evaluate_segmentation(model: torch.nn.Module,
                          loader,
                          num_classes: int,
                          device: torch.device,
                          ignore_index: Optional[int] = None
                         ) -> Tuple[float, float, float]:
    """
    Повертає:
      val_loss (CE), pixel_acc, mIoU.
    """
    model.eval()
    hist = torch.zeros((num_classes, num_classes), device=device, dtype=torch.float32)


    ce = torch.nn.CrossEntropyLoss(ignore_index=ignore_index) if ignore_index is not None \
         else torch.nn.CrossEntropyLoss()

    val_loss = 0.0
    n_batches = 0

    for imgs, masks in loader:
        imgs  = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        logits = model(imgs)
        loss = ce(logits, masks)
        val_loss += float(loss.item())
        n_batches += 1

        preds = torch.argmax(logits, dim=1)


        if preds.dim() == 3:
            for p, t in zip(preds, masks):

                if ignore_index is not None:
                    t = t.clone()
                    t[t == ignore_index] = -1
                hist += _fast_hist(p, t, num_classes)
        else:

            if ignore_index is not None:
                masks = masks.clone()
                masks[masks == ignore_index] = -1
            hist += _fast_hist(preds, masks, num_classes)

    val_loss /= max(1, n_batches)


    total = hist.sum().clamp_min(1.0)
    acc = torch.diag(hist).sum() / total


    diag = torch.diag(hist)
    denom = (hist.sum(1) + hist.sum(0) - diag).clamp_min(1.0)
    iu = diag / denom


    miou = iu.mean()

    return val_loss, float(acc.item()), float(miou.item())


In [ ]:

import os
from torch.nn.utils import clip_grad_norm_
from torch.cuda.amp import autocast, GradScaler

use_amp = (cfg.device.type == "cuda")
scaler = GradScaler(enabled=use_amp)

def train_fcn_one_epoch(model, loader, optimizer, device, grad_clip=None, ignore_index=None):
    model.train()
    ce = torch.nn.CrossEntropyLoss(ignore_index=ignore_index)
    running = 0.0
    n_batches = 0

    for imgs, masks in loader:
        imgs  = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=use_amp):
            logits = model(imgs)
            loss = ce(logits, masks)

        if use_amp:
            scaler.scale(loss).backward()
            if grad_clip:
                scaler.unscale_(optimizer)
                clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if grad_clip:
                clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        running += float(loss.item())
        n_batches += 1

    return running / max(1, n_batches)


hp_fcn = hp_compose("fcn32s")

train_ds_sem = COCOSemanticDataset(
    cfg.train_root, cfg.train_ann,
    hflip_p=DEFAULTS_MINI["aug"]["hflip_p"]
)
val_ds_sem   = COCOSemanticDataset(
    cfg.val_root, cfg.val_ann,
    hflip_p=0.0
)

pin = (cfg.device.type == "cuda")
train_loader_sem = DataLoader(train_ds_sem,
                              batch_size=hp_fcn["train"]["batch_size"],
                              shuffle=True, num_workers=2,
                              pin_memory=pin)
val_loader_sem   = DataLoader(val_ds_sem,
                              batch_size=hp_fcn["train"]["batch_size"],
                              shuffle=False, num_workers=2,
                              pin_memory=pin)

num_classes_sem = train_ds_sem.num_classes
model_fcn = FCN32s(num_classes=num_classes_sem, pretrained_backbone=True).to(cfg.device)
opt_fcn   = build_optimizer(model_fcn, hp_fcn)
sch_fcn   = build_scheduler(opt_fcn, hp_fcn)

os.makedirs(cfg.checkpoint_dir, exist_ok=True)
best_miou = -1.0

hist_fcn = History()
for epoch in range(1, hp_fcn["train"]["epochs"] + 1):
    tr_loss = train_fcn_one_epoch(
        model_fcn, train_loader_sem, opt_fcn, cfg.device,
        grad_clip=hp_fcn["train"]["grad_clip"],
        ignore_index=None
    )
    val_loss, pixacc, miou = evaluate_segmentation(
        model_fcn, val_loader_sem, num_classes_sem, cfg.device,
        ignore_index=None
    )
    if sch_fcn:
        sch_fcn.step()


    if miou > best_miou:
        best_miou = miou
        torch.save(model_fcn.state_dict(),
                   os.path.join(cfg.checkpoint_dir, "fcn32s_best.pt"))

    hist_fcn.log(epoch=epoch, train_loss=tr_loss, val_loss=val_loss,
                 metric=miou, pixel_acc=pixacc, miou=miou)
    print(f"[Epoch {epoch:02d}] loss_tr={tr_loss:.4f}  loss_val={val_loss:.4f}  Acc={pixacc:.3f}  mIoU={miou:.3f}")

df_hist_fcn = hist_fcn.to_df()
summarize_run(df_hist_fcn, hp_fcn, save_json_path="best_hparams_fcn32s.json")


In [ ]:

from __future__ import annotations
import json
from pathlib import Path
from typing import Tuple, Dict, Any, Optional

import pandas as pd
import matplotlib.pyplot as plt



def extract_best_hp_from_results(df: pd.DataFrame, model_key: str
                                 ) -> Tuple[Dict[str, Any], Dict[str, Any], Dict[str, Any]]:
    """
    Формує гіперпараметри з першого (найкращого) рядка results DataFrame.
    Очікується, що df уже відсортований за 'metric' (спадаюче).
    """
    if df is None or df.empty:
        raise ValueError("Порожній results DataFrame.")

    top = df.iloc[0].to_dict()
    overrides_train = {k.split('.', 1)[1]: v for k, v in top.items() if k.startswith('train.')}
    overrides_model = {k.split('.', 1)[1]: v for k, v in top.items() if k.startswith(model_key + '.')}
    hp = hp_compose(model_key, overrides={'train': overrides_train, model_key: overrides_model})
    return hp, overrides_train, overrides_model


def _safe_display(obj) -> None:
    """display() доступний не всюди — робимо безпечний виклик."""
    try:
        from IPython.display import display as ipy_display
        ipy_display(obj)
    except Exception:

        try:
            print(obj.head() if hasattr(obj, "head") else str(obj))
        except Exception:
            pass


def plot_history(df_hist: pd.DataFrame, metric_col: Optional[str] = 'metric',
                 title: str = 'Training / Validation'):
    """
    Малює криві train/val та (опційно) метрику ↑.
    Повертає (fig, ax). Не викликає plt.close().
    """
    if df_hist is None or df_hist.empty:
        raise ValueError("df_hist порожній — нічого малювати.")


    dfp = (df_hist
           .drop_duplicates(subset=['epoch'], keep='last')
           .sort_values('epoch')
           .reset_index(drop=True))

    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    if 'train_loss' in dfp.columns:
        ax.plot(dfp['epoch'], pd.to_numeric(dfp['train_loss'], errors='coerce'), label='train_loss')
    if 'val_loss' in dfp.columns:
        ax.plot(dfp['epoch'], pd.to_numeric(dfp['val_loss'], errors='coerce'), label='val_loss')
    if metric_col and metric_col in dfp.columns:
        ax.plot(dfp['epoch'], pd.to_numeric(dfp[metric_col], errors='coerce'), label=metric_col)

    ax.set_xlabel('epoch')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    return fig, ax


def summarize_run(df_hist: pd.DataFrame,
                  best_hp: Dict[str, Any],
                  save_json_path: str = 'best_hparams.json',
                  save_plot_path: Optional[str] = None,
                  metric_prefers_higher: bool = True) -> Optional[pd.Series]:
    """
    Підсумок навчання:
      - показує останні рядки історії
      - малює графік і (опційно) зберігає його
      - зберігає best_hp у JSON
      - знаходить найкращий епох (за 'metric' або мінімальним 'val_loss')
    Повертає рядок best_row (pd.Series) або None.
    """
    if df_hist is None or df_hist.empty:
        raise ValueError("df_hist порожній.")


    if 'metric' in df_hist.columns:
        metric_col = 'metric'
        prefers_higher = metric_prefers_higher
    elif 'val_loss' in df_hist.columns:
        metric_col = 'val_loss'
        prefers_higher = False
    else:
        metric_col = None
        prefers_higher = True

    print("=== ПІДСУМОК НАВЧАННЯ ===")
    _safe_display(df_hist.tail())


    fig, ax = plot_history(df_hist, metric_col=('metric' if 'metric' in df_hist.columns else None))
    if save_plot_path:
        save_plot_path = str(save_plot_path)
        Path(save_plot_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_plot_path, dpi=150)
    plt.show()

    print("\n=== Найкращі гіперпараметри ===")

    safe_hp = json.loads(json.dumps(best_hp, default=str))
    print(json.dumps(safe_hp, indent=2, ensure_ascii=False))

    save_json_path = str(save_json_path)
    Path(save_json_path).parent.mkdir(parents=True, exist_ok=True)
    with open(save_json_path, 'w', encoding='utf-8') as f:
        json.dump(safe_hp, f, indent=2, ensure_ascii=False)

    best_row = None
    if metric_col is not None:
        s = pd.to_numeric(df_hist[metric_col], errors='coerce')
        if s.notna().any():
            idx = s.idxmax() if prefers_higher else s.idxmin()
            best_row = df_hist.loc[idx]
            print("\n=== Найкращий епох ===")
            _safe_display(pd.DataFrame([best_row]))
        else:
            print("\nПопередження: усі значення метрики NaN — неможливо визначити найкращий епох.")
    else:
        print("\nПопередження: немає колонок 'metric'/'val_loss' — пропускаю вибір найкращого епоху.")

    return best_row


In [ ]:

import random
from typing import Optional, Dict, Any

def quick_train_stub(model,
                     hp: Dict[str, Any],
                     epochs: Optional[int] = None,
                     noise: float = 0.02,
                     base_loss: float = 1.0):
    """
    Синтетичне тренування для відладки пайплайну.
    Повертає DataFrame з колонками: epoch, train_loss, val_loss, metric (↑ краще), lr.

    Параметри:
      - model: будь-який nn.Module (не використовується, лише для інтерфейсу).
      - hp: словник гіперпараметрів у форматі hp_compose(...).
      - epochs: кількість епох (за замовчуванням hp['train']['epochs'] або 10).
      - noise: амплітуда випадкового шуму у втраті.
      - base_loss: стартове значення train loss (перед «покращенням»).

    Примітки:
      - «metric» визначено як 1.0 - val_loss (тобто чим менше val_loss, тим краща метрика).
      - Логіку шедулера імітуємо локально (StepLR_5_0.5).
    """

    train_hp = hp.get("train", {})
    seed = int(train_hp.get("seed", 42))
    epochs = int(epochs if epochs is not None else train_hp.get("epochs", 10))
    start_lr = float(train_hp.get("learning_rate", 1e-3))
    scheduler_name = str(train_hp.get("scheduler", "none"))

    rng = random.Random(seed)

    history = History()


    decay_per_epoch = 0.06

    for epoch in range(1, epochs + 1):

        if scheduler_name == "StepLR_5_0.5":

            step_count = (epoch - 1) // 5
            lr = start_lr * (0.5 ** step_count)
        else:
            lr = start_lr


        train_loss = max(0.0, base_loss - decay_per_epoch * epoch) + rng.uniform(0.0, noise)
        val_loss   = train_loss + 0.05 + rng.uniform(0.0, noise * 0.5)


        metric = 1.0 - val_loss

        history.log(
            epoch=epoch,
            train_loss=float(train_loss),
            val_loss=float(val_loss),
            metric=float(metric),
            lr=float(lr),
        )

    return history.to_df()
